In [65]:
# ==================================================
# Configuració del projecte
# ==================================================

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [67]:
# ==================================================
# Imports
# ==================================================

import pandas as pd

from src.config import (
    DATA_RAW,
    DATA_INTERIM,
    DATA_PROCESSED,
)

from src.utils_io import (
    load_csv,
    save_csv,
)

from src.harmonization import (
    clean_year_columns,
    detect_year_columns,
    harmonize_territory,
    melt_years,
    normalize_text,
    remove_aei_suffix,
)

In [69]:
# ==================================================
# Carregar datasets clean
# ==================================================

print("Carregant datasets INTERIM...")

FILES = {
    "rent": (
        "rent_clean.csv",
        {"thousands": ".", "decimal": ","},
    ),

    "income": (
        "income_clean.csv",
        {},
    ),

    "nat": (
        "nationality_clean.csv",
        {"thousands": ".", "decimal": ","},
    ),

    "pop": (
        "population_clean.csv",
        {"thousands": ".", "decimal": ","},
    ),

    "atur": (
        "atur_clean.csv",
        {"thousands": ".", "decimal": ","},
    ),

    "est": (
        "estudis_clean.csv",
        {"thousands": ".", "decimal": ","},
    ),

    "cont": (
        "contractes_clean.csv",
        {"thousands": ".", "decimal": ","},
    ),

    "sup": (
        "superficie_clean.csv",
        {"thousands": ".", "decimal": ","},
    ),

    "hut": (
        "turisme_clean.csv",
        {"thousands": ".", "decimal": ","},
    ),
}

dfs = {
    key: load_csv(
        DATA_INTERIM / filename,
        **params,
    )

    for key, (filename, params)
    in FILES.items()
}

print("Datasets carregats correctament.")

Carregant datasets INTERIM...
Datasets carregats correctament.


In [71]:
# ==================================================
# Harmonització territorial
# ==================================================

TERRITORY_MAP = {

    # Exemple:
    # "el poble sec": "poble sec",

}

def normalize_territory_block(df):

    df = df.copy()

    df["territori"] = (
        df["territori"]
        .apply(normalize_text)
        .apply(remove_aei_suffix)
    )

    df = harmonize_territory(
        df,
        "territori",
        TERRITORY_MAP,
    )

    return df

for key in dfs:

    dfs[key] = normalize_territory_block(
        dfs[key]
    )

print("Territoris harmonitzats.")


Territoris harmonitzats.


In [73]:
# ==================================================
# Detecció de columnes especials
# ==================================================

nat_col = next(
    c for c in dfs["nat"].columns
    if "nacionalitat" in c
)

tit_col = next(
    (
        c for c in dfs["est"].columns
        if (
            "titul" in c
            or "estudi" in c
            or "escolar" in c
        )
    ),
    None
)

print("Columna titulació:", tit_col)
print("Columna nacionalitat:", nat_col)

Columna titulació: titulació_acadèmica
Columna nacionalitat: nacionalitat_espanya_ue_resta_estranger


In [75]:
 # ==================================================
# Neteja columnes d'any
# ==================================================

for key in [
    "pop",
    "nat",
    "atur",
    "est",
    "hut",
]:

    dfs[key] = clean_year_columns(
        dfs[key]
    )

print("Columnes d'any harmonitzades.")

Columnes d'any harmonitzades.


In [77]:
# ==================================================
# Funció auxiliar LONG
# ==================================================

def build_long_dataset(
    df,
    value_name,
    id_vars=None,
):

    if id_vars is None:

        id_vars = [
            "territori",
            "tipus_de_territori",
        ]

    return melt_years(
        df,
        id_vars=id_vars,
        value_name=value_name,
    )

In [79]:
# ==================================================
# Conversió datasets a format LONG
# ==================================================

print("Convertint datasets a format LONG...")

# --- LLOGUER

df_rent_long = build_long_dataset(
    dfs["rent"],
    value_name="preu_lloguer",
)

# --- RENDA

df_income_long = dfs["income"].rename(
    columns={
        "areatype": "tipus_de_territori",
        "value": "renda",
        "year": "any",
    }
)

# --- NACIONALITAT

df_nat_long = build_long_dataset(
    dfs["nat"],
    value_name="poblacio",
    id_vars=[
        "territori",
        "tipus_de_territori",
        nat_col,
    ],
)

# --- POBLACIÓ

df_pop_long = build_long_dataset(
    dfs["pop"],
    value_name="poblacio_total",
)

# --- ATUR

df_atur_long = build_long_dataset(
    dfs["atur"],
    value_name="atur",
)

# --- ESTUDIS

df_est_long = build_long_dataset(
    dfs["est"],
    value_name="poblacio",
    id_vars=[
        "territori",
        "tipus_de_territori",
        tit_col,
    ],
)

# --- CONTRACTES

df_cont_long = build_long_dataset(
    dfs["cont"],
    value_name="num_contractes",
)

# --- SUPERFÍCIE

df_sup_long = build_long_dataset(
    dfs["sup"],
    value_name="superficie_mitja",
)

# --- TURISME

df_hut_long = build_long_dataset(
    dfs["hut"],
    value_name="hut",
)

print("Conversió LONG completada.")

Convertint datasets a format LONG...
Conversió LONG completada.


In [81]:
# ==================================================
# Validació ràpida
# ==================================================

print(df_hut_long.shape)

display(
    df_hut_long.head()
)

print(
    df_hut_long.isna().sum()
)

(715, 4)


,territori,tipus_de_territori,any,hut
0,el raval,Barri,2014,180.0
1,el barri gotic,Barri,2014,184.0
2,la barceloneta,Barri,2014,69.0
3,sant pere santa caterina i la ribera,Barri,2014,171.0
4,el fort pienc,Barri,2014,343.0


territori              0
tipus_de_territori     0
any                    0
hut                   11
dtype: int64


In [83]:
# ==================================================
# Exportació datasets harmonitzats
# ==================================================

print("Guardant datasets harmonitzats...")

OUTPUTS = {

    "rent_harmonized.csv":
        df_rent_long,

    "income_harmonized.csv":
        df_income_long,

    "nationality_harmonized.csv":
        df_nat_long,

    "population_harmonized.csv":
        df_pop_long,

    "atur_harmonized.csv":
        df_atur_long,

    "estudis_harmonized.csv":
        df_est_long,

    "contractes_harmonized.csv":
        df_cont_long,

    "superficie_harmonized.csv":
        df_sup_long,

    "hut_harmonized.csv":
        df_hut_long,
}

for filename, dataset in OUTPUTS.items():

    save_csv(
        dataset,
        DATA_PROCESSED / filename,
    )

print(
    "Notebook 02_harmonize "
    "completat correctament."
)

Guardant datasets harmonitzats...
Notebook 02_harmonize completat correctament.
